# Iceberg Demo — `events` & `logging` tables

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.remote(
    os.environ.get("SPARK_REMOTE", "sc://spark-connect:15002")
).getOrCreate()

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

## Events table

### 1. Create table, partitioned by `hour(event_ts), event_name`

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.events")
spark.sql("""
    CREATE TABLE default.events (
        event_id    INT,
        event_name  STRING,
        event_ts    TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (hour(event_ts), event_name)
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    """)
print("✅ Created table `default.events`, partitioned by hour(event_ts)")

In [ ]:
spark.sql("""
    ALTER TABLE default.events
    WRITE ORDERED BY (event_ts, event_name)
    """)

### 2. Insert data for hours 10 and 11

In [ ]:
hours_data = [
    (333, "corrected_login", "2025-06-15 10:05:00"),
    (2, "click", "2025-06-15 11:20:00"),
    (23, "corrected_login", "2025-06-15 10:45:00"),
]

df_hours = spark.createDataFrame(hours_data, ["event_id", "event_name", "event_ts_str"])
df_hours = df_hours.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_hours.writeTo("default.events").overwritePartitions()

print("✅ Inserted 3 rows into hours 10 and 11 partition")

spark.table("default.events").show(1000, False)

### 2.5 Schema evolution: add a column, rename one

In [ ]:
print("📋 Schema before evolution:")
spark.table("default.events").printSchema()

spark.sql("ALTER TABLE default.events ADD COLUMNS (event_source STRING)")
print("✅ Added column: event_source")

spark.sql("ALTER TABLE default.events RENAME COLUMN event_name TO event_type")
print("✅ Renamed column: event_name → event_type")

print("\n📋 Schema after evolution:")
spark.table("default.events").printSchema()
spark.table("default.events").show(1000, False)

### 3. Insert data for hour 11 (11:00–11:59) using the new schema

In [ ]:
hour11_data = [
    (4, "logout", "2025-06-15 11:10:00", "mobile"),
    (5, "signup", "2025-06-15 11:30:00", "web"),
]
df_hour11 = spark.createDataFrame(hour11_data, ["event_id", "event_type", "event_ts_str", "event_source"])
df_hour11 = df_hour11.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_hour11.writeTo("default.events").append()
df_hour11.writeTo("default.events").append()
print("✅ Inserted 2 rows into hour 11 partition (with new columns)")
spark.table("default.events").show(1000, False)

### 4. Overwrite ONLY the hour-10 partition with replacement data

In [ ]:
hour10_replacement = [
    (100, "corrected_login", "2025-06-15 10:05:00", "api"),
    (101, "corrected_purchase", "2025-06-15 10:45:00", "api"),
]
df_hour10_new = spark.createDataFrame(hour10_replacement, ["event_id", "event_type", "event_ts_str", "event_source"])
df_hour10_new = df_hour10_new.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_hour10_new.writeTo("default.events").overwritePartitions()
print("✅ Overwrote hour 10 partition with corrected data")
print("   (Hour 11 remains untouched)\n")

print("📋 Final table contents:")
spark.table("default.events").show(1000, False)

### 5. Create a branch from the current snapshot

In [ ]:
spark.sql("ALTER TABLE default.events CREATE BRANCH my_test_branch")

### 6. Write data to the branch

In [ ]:
branch_data = [
    (300, "audit_fix", "2025-06-15 13:15:00", "audit"),
    (301, "audit_review", "2025-06-15 13:45:00", "audit"),
    (301, "audit_review", "2025-06-15 13:45:00", "audit"),
]
df_branch = spark.createDataFrame(branch_data, ["event_id", "event_type", "event_ts_str", "event_source"])
df_branch = df_branch.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_branch.writeTo("default.events.branch_my_test_branch").overwritePartitions()
print("✅ Overwrote hour 13 partition on my_test_branch")

### 6.5 Write data to `main` that won't appear on the branch (with a custom snapshot property)

In [ ]:
experimental_data = [
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment1", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:03:00", "test"),
    (200, "experiment", "2025-06-15 12:30:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment2", "2025-06-15 12:10:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:00:00", "test"),
    (200, "experiment", "2025-06-15 12:01:00", "test"),
]

df_exp = spark.createDataFrame(experimental_data, ["event_id", "event_type", "event_ts_str", "event_source"])

df_exp = df_exp.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")

df_exp.writeTo("default.events").option(
    "snapshot-property.my_custom_key",
    "If you have found me, you are searching through IceGraph and doing a great job!",
).overwritePartitions()

print("✅ Overwrote hour 12 partition on main (after branching)")

### 7. Compare `main` vs `my_test_branch`

In [ ]:
print("📋 Main branch:")
spark.table("default.events").show(1000, False)

print("📋 my_test_branch:")
spark.read.option("branch", "my_test_branch").table("default.events").show(1000, False)

### 8. Create a second branch `my_merge_branch`, this one gets merged back

This branch is created from the current `main` snapshot and, unlike `my_test_branch`, nothing is
written to `main` afterwards, so it can be merged back cleanly at the end.

In [ ]:
spark.sql("ALTER TABLE default.events DROP BRANCH IF EXISTS my_merge_branch")
spark.sql("ALTER TABLE default.events CREATE BRANCH my_merge_branch")
print("✅ Created branch `my_merge_branch` from the current main snapshot")

### 9. Write to `my_merge_branch` (two commits, `main` stays untouched)

In [ ]:
merge_branch_data = [
    (400, "checkout", "2025-06-15 14:05:00", "web"),
    (401, "checkout", "2025-06-15 14:35:00", "mobile"),
    (402, "checkout_failed", "2025-06-15 14:20:00", "web"),
]
df_merge_branch = spark.createDataFrame(merge_branch_data, ["event_id", "event_type", "event_ts_str", "event_source"])
df_merge_branch = df_merge_branch.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_merge_branch.writeTo("default.events.branch_my_merge_branch").append()
print("✅ First commit on my_merge_branch: appended 3 rows into hour 14")

In [ ]:
merge_branch_fix = [
    (400, "checkout", "2025-06-15 14:05:00", "web"),
    (402, "checkout_retry", "2025-06-15 14:20:00", "web"),
    (403, "refund", "2025-06-15 15:05:00", "support"),
]
df_merge_fix = spark.createDataFrame(merge_branch_fix, ["event_id", "event_type", "event_ts_str", "event_source"])
df_merge_fix = df_merge_fix.withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
df_merge_fix.writeTo("default.events.branch_my_merge_branch").overwritePartitions()
print("✅ Second commit on my_merge_branch: overwrote the hour-14 partitions, added hour 15")

spark.read.option("branch", "my_merge_branch").table("default.events").show(1000, False)

In [ ]:
print("📋 main (no hour 14 / 15 yet):")
spark.table("default.events").show(1000, False)

print("📋 my_merge_branch (hours 14 and 15 only live here):")
spark.read.option("branch", "my_merge_branch").table("default.events").show(1000, False)

### 10. Merge `my_merge_branch` back into `main`

`main` has not moved since `my_merge_branch` was created, so `main` is an ancestor of the branch
and the merge is a **fast-forward**: `main` simply moves to the branch tip and picks up both branch snapshots.

Notes:
* `my_test_branch` (created in step 5) **cannot** be fast-forwarded, because `main` moved after it was
  created (the hour-12 write in step 6.5). That diverged branch is merged in step 11 with a cherry-pick.
* The branch ref still exists after the fast-forward, it just points at the same snapshot as `main`.
  Both branches are dropped in step 12.

In [ ]:
print("📋 main BEFORE the merge:")
spark.table("default.events").show(1000, False)

spark.sql("""
    CALL spark_catalog.system.fast_forward('default.events', 'main', 'my_merge_branch')
""").show(truncate=False)
print("✅ Fast-forwarded `main` to `my_merge_branch`")

print("📋 main AFTER the merge (hours 14 and 15 are now on main):")
spark.table("default.events").show(1000, False)

print("📋 my_test_branch is untouched by the merge:")
spark.read.option("branch", "my_test_branch").table("default.events").show(1000, False)

### 11. Merge the diverged `my_test_branch` into `main` (the harder case)

`main` moved after `my_test_branch` was created (the hour-12 write in step 6.5, plus the fast-forward
in step 10), so the two histories diverged and `fast_forward` would fail. Instead we **cherry-pick** the
branch commit: Iceberg replays that snapshot on top of the current `main`, so `main` keeps everything it
already had and gains the hour-13 rows that only lived on the branch.

Notes:
* Only `append` and dynamic-overwrite (`overwritePartitions`) snapshots can be cherry-picked, and each
  call replays exactly one snapshot, so a branch with several commits needs one call per commit,
  oldest first. That is why `my_merge_branch` (two commits) was fast-forwarded instead.
* For arbitrary divergence, the general fallback is a row-level
  `MERGE INTO default.events t USING default.events.branch_my_test_branch s ON ...`.

In [ ]:
refs = spark.table("default.events.refs")
print("📋 Refs before the cherry-pick:")
refs.show(truncate=False)

test_branch_snapshot = refs.where("name = 'my_test_branch'").select("snapshot_id").first()[0]
print(f"my_test_branch head snapshot: {test_branch_snapshot}")

spark.sql(f"""
    CALL spark_catalog.system.cherrypick_snapshot('default.events', {test_branch_snapshot})
""").show(truncate=False)
print("✅ Cherry-picked the my_test_branch commit onto main")

print("📋 main after both merges (hours 12, 13, 14 and 15 are all there):")
spark.table("default.events").show(1000, False)

### 12. Clean up: drop both branches

Both branches are merged into `main` now, so the refs can go. Dropping a branch only removes the
reference, the snapshots themselves stay in the table history until `expire_snapshots` runs.

In [ ]:
spark.sql("ALTER TABLE default.events DROP BRANCH my_merge_branch")
spark.sql("ALTER TABLE default.events DROP BRANCH my_test_branch")
print("✅ Dropped `my_merge_branch` and `my_test_branch`")

print("📋 Remaining refs (only `main` is left):")
spark.table("default.events.refs").show(truncate=False)

print("📋 Final main contents:")
spark.table("default.events").show(1000, False)

## Logging table

### 1. Create table with a deeply nested schema, partitioned by `hour(event_ts)`

This one is deliberately extreme: a struct, a struct inside a struct inside a struct, an array,
an array of structs (each with its own map), three maps (string values, double values, struct values)
and an array of arrays.

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.logging")
spark.sql("""
    CREATE TABLE default.logging (
        event_id    INT,
        event_name  STRING,
        event_ts    TIMESTAMP,
        user_info   STRUCT<
                        id: BIGINT,
                        name: STRING,
                        roles: ARRAY<STRING>,
                        address: STRUCT<
                            city: STRING,
                            country: STRING,
                            geo: STRUCT<lat: DOUBLE, lon: DOUBLE>
                        >
                    >,
        tags        ARRAY<STRING>,
        items       ARRAY<STRUCT<
                        sku: STRING,
                        qty: INT,
                        price: DECIMAL(10,2),
                        attrs: MAP<STRING, STRING>
                    >>,
        attributes  MAP<STRING, STRING>,
        metrics     MAP<STRING, DOUBLE>,
        sessions    MAP<STRING, STRUCT<started_at_ms: BIGINT, duration_ms: BIGINT>>,
        matrix      ARRAY<ARRAY<INT>>
    )
    USING iceberg
    PARTITIONED BY (hour(event_ts))
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    """)
print("✅ Created table `default.logging`, partitioned by hour(event_ts)")

spark.table("default.logging").printSchema()

### 2. Insert data for hour 10 (10:00–10:59)

In [ ]:
from datetime import datetime
from decimal import Decimal

# Rows are plain Python tuples: tuple -> struct, list -> array, dict -> map.
# Field order follows the table schema, which is read back here instead of being retyped.
logging_schema = spark.table("default.logging").schema

hour10_data = [
    (
        1,
        "login",
        datetime(2025, 6, 15, 10, 5),
        (10, "Ada Lovelace", ["admin", "analyst"], ("London", "UK", (51.5074, -0.1278))),
        ["auth", "web"],
        [("SKU-1", 2, Decimal("19.99"), {"color": "red", "size": "M"})],
        {"channel": "web", "ab_test": "B"},
        {"latency_ms": 120.5, "success_rate": 1.0},
        {"sess-a1": (1750000000000, 4500)},
        [[1, 2], [3, 4]],
    ),
    (
        2,
        "click",
        datetime(2025, 6, 15, 10, 20),
        (11, "Grace Hopper", ["viewer"], ("New York", "US", (40.7128, -74.0060))),
        ["ui"],
        [],
        {"channel": "mobile"},
        {"latency_ms": 45.0},
        {"sess-b2": (1750000100000, 900)},
        [[0]],
    ),
    (
        3,
        "purchase",
        datetime(2025, 6, 15, 10, 45),
        (12, "Alan Turing", ["buyer", "viewer"], ("Cambridge", "UK", (52.2053, 0.1218))),
        ["checkout", None],
        [
            ("SKU-2", 1, Decimal("249.00"), {"gift": "true"}),
            ("SKU-3", 5, Decimal("3.50"), {}),
        ],
        {"channel": "web", "coupon": "SUMMER"},
        {"latency_ms": 310.25, "cart_value": 266.5},
        {"sess-c3": (1750000200000, 15000), "sess-c4": (1750000900000, 300)},
        [[7, 8, 9], [10], []],
    ),
]

df_hour10 = spark.createDataFrame(hour10_data, logging_schema)
df_hour10.writeTo("default.logging").overwritePartitions()

print("✅ Inserted 3 rows into hour 10 partition")

spark.table("default.logging").show(1000, False)

### 2.5 Schema evolution: top-level, nested, and inside an array element

Iceberg can evolve nested types by path, so this step adds a field to a struct, adds a field to the
struct inside `items`, renames both a top-level column and a nested one, and widens `event_id` to BIGINT.
All of it is metadata-only, no data is rewritten.

In [ ]:
print("📋 Schema before evolution:")
spark.table("default.logging").printSchema()

spark.sql("ALTER TABLE default.logging ADD COLUMNS (event_source STRING)")
print("✅ Added top-level column: event_source")

spark.sql("ALTER TABLE default.logging ADD COLUMNS (user_info.email STRING)")
print("✅ Added nested field: user_info.email")

spark.sql("ALTER TABLE default.logging ADD COLUMNS (items.element.discount DOUBLE)")
print("✅ Added field inside the array element struct: items.element.discount")

spark.sql("ALTER TABLE default.logging RENAME COLUMN event_name TO event_type")
print("✅ Renamed column: event_name → event_type")

spark.sql("ALTER TABLE default.logging RENAME COLUMN user_info.name TO full_name")
print("✅ Renamed nested field: user_info.name → user_info.full_name")

spark.sql("ALTER TABLE default.logging ALTER COLUMN event_id TYPE bigint")
print("✅ Widened type: event_id INT → BIGINT")

print("\n📋 Schema after evolution:")
spark.table("default.logging").printSchema()
spark.table("default.logging").show(1000, False)

### 3. Insert data for hour 11 (11:00–11:59) using the evolved schema

New fields land at the end of their parent struct, so `user_info` is now
`(id, full_name, roles, address, email)` and an `items` element is `(sku, qty, price, attrs, discount)`.

In [ ]:
logging_schema_v2 = spark.table("default.logging").schema

hour11_data = [
    (
        4,
        "logout",
        datetime(2025, 6, 15, 11, 10),
        (10, "Ada Lovelace", ["admin"], ("London", "UK", (51.5074, -0.1278)), "ada@example.com"),
        ["auth"],
        [],
        {"channel": "mobile"},
        {"latency_ms": 22.0},
        {"sess-a1": (1750003600000, 60)},
        [[1]],
        "mobile",
    ),
    (
        5,
        "signup",
        datetime(2025, 6, 15, 11, 30),
        (13, "Katherine Johnson", ["viewer", "buyer"], ("Hampton", "US", (37.0299, -76.3452)), "kj@example.com"),
        ["growth", "web"],
        [("SKU-9", 1, Decimal("0.00"), {"trial": "true"}, 100.0)],
        {"channel": "web", "referrer": "newsletter"},
        {"latency_ms": 88.5, "score": 0.73},
        {"sess-d5": (1750004000000, 12000)},
        [[2, 3], [4, 5]],
        "web",
    ),
]

df_hour11 = spark.createDataFrame(hour11_data, logging_schema_v2)
df_hour11.writeTo("default.logging").append()
print("✅ Inserted 2 rows into hour 11 partition (with the new nested fields)")
spark.table("default.logging").show(1000, False)

### 4. Overwrite ONLY the hour-10 partition with replacement data

In [ ]:
hour10_replacement = [
    (
        100,
        "corrected_login",
        datetime(2025, 6, 15, 10, 5),
        (10, "Ada Lovelace", ["admin", "analyst"], ("London", "UK", (51.5074, -0.1278)), "ada@example.com"),
        ["auth", "corrected"],
        [],
        {"channel": "api", "fix": "replay"},
        {"latency_ms": 99.9},
        {"sess-a1": (1750000000000, 4500)},
        [[1, 1], [2, 2]],
        "api",
    ),
    (
        101,
        "corrected_login",
        datetime(2025, 6, 15, 10, 45),
        (12, "Alan Turing", ["buyer"], ("Cambridge", "UK", (52.2053, 0.1218)), "alan@example.com"),
        ["auth"],
        [("SKU-2", 1, Decimal("249.00"), {"gift": "true"}, 10.0)],
        {"channel": "api"},
        {"latency_ms": 150.0},
        {},
        [[]],
        "api",
    ),
]

df_hour10_new = spark.createDataFrame(hour10_replacement, spark.table("default.logging").schema)
df_hour10_new.writeTo("default.logging").overwritePartitions()

print("✅ Overwrote hour 10 partition with corrected data")
print("   (Hour 11 remains untouched)\n")

print("📋 Final table contents:")
spark.table("default.logging").show(1000, False)

# Seeing the change

To see the changes you did to your tables in a clear and easy way, check https://github.com/YanivZalach/IceGraph

## Views

### Create a view on top of the `logging` table

In [ ]:
spark.sql("""
    CREATE OR REPLACE VIEW default.logging_summary AS
    SELECT
        event_type,
        event_source,
        user_info.address.country                AS country,
        count(*)                                 AS event_count,
        sum(coalesce(size(items), 0))            AS item_count,
        max(metrics['latency_ms'])               AS max_latency_ms
    FROM default.logging
    GROUP BY event_type, event_source, user_info.address.country
""")
print("✅ Created view `default.logging_summary` over `default.logging`")

spark.table("default.logging_summary").show(1000, False)

# The nested columns are queryable straight from the table too.
spark.sql("""
    SELECT event_id, user_info.full_name, item.sku, item.qty, item.price, tag
    FROM default.logging
    LATERAL VIEW OUTER explode(items) AS item
    LATERAL VIEW OUTER explode(tags)  AS tag
    ORDER BY event_id
""").show(1000, False)

## Orders table

### 1. Create table, partitioned by `days(order_ts)`

A normal, everyday table: flat columns plus a single small struct for the shipping address.

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.orders")
spark.sql("""
    CREATE TABLE default.orders (
        order_id   BIGINT,
        customer   STRING,
        amount     DECIMAL(10,2),
        order_ts   TIMESTAMP,
        shipping   STRUCT<city: STRING, country: STRING>
    )
    USING iceberg
    PARTITIONED BY (days(order_ts))
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    """)
print("✅ Created table `default.orders`, partitioned by days(order_ts)")

spark.table("default.orders").printSchema()

### 2. First write: append 3 orders

In [ ]:
from datetime import datetime
from decimal import Decimal

orders_day1 = [
    (1001, "Ada Lovelace", Decimal("120.50"), datetime(2025, 6, 15, 9, 30), ("London", "UK")),
    (1002, "Grace Hopper", Decimal("89.99"), datetime(2025, 6, 15, 14, 5), ("New York", "US")),
    (1003, "Alan Turing", Decimal("249.00"), datetime(2025, 6, 15, 18, 45), ("Cambridge", "UK")),
]

df_orders = spark.createDataFrame(orders_day1, spark.table("default.orders").schema)
df_orders.writeTo("default.orders").append()

print("✅ Appended 3 orders for 2025-06-15")
spark.table("default.orders").show(1000, False)

### 3. Schema change 1: add a `status` column

In [ ]:
spark.sql("ALTER TABLE default.orders ADD COLUMNS (status STRING)")
print("✅ Added column: status")

spark.table("default.orders").printSchema()
spark.table("default.orders").show(1000, False)

### 4. Second write: append 2 more orders, now with `status`

The rows written before the change keep a NULL `status`, no data is rewritten.

In [ ]:
orders_day2 = [
    (1004, "Katherine Johnson", Decimal("35.00"), datetime(2025, 6, 16, 10, 15), ("Hampton", "US"), "shipped"),
    (1005, "Ada Lovelace", Decimal("410.75"), datetime(2025, 6, 16, 16, 40), ("London", "UK"), "pending"),
]

df_orders_day2 = spark.createDataFrame(orders_day2, spark.table("default.orders").schema)
df_orders_day2.writeTo("default.orders").append()

print("✅ Appended 2 orders for 2025-06-16")
spark.table("default.orders").show(1000, False)

### 5. Schema change 2: rename `customer` to `customer_name`

In [ ]:
spark.sql("ALTER TABLE default.orders RENAME COLUMN customer TO customer_name")
print("✅ Renamed column: customer → customer_name")

spark.table("default.orders").printSchema()

### 6. Third write: overwrite only the 2025-06-15 partition

In [ ]:
orders_day1_fixed = [
    (1001, "Ada Lovelace", Decimal("130.50"), datetime(2025, 6, 15, 9, 30), ("London", "UK"), "delivered"),
    (1003, "Alan Turing", Decimal("249.00"), datetime(2025, 6, 15, 18, 45), ("Cambridge", "UK"), "returned"),
]

df_orders_fixed = spark.createDataFrame(orders_day1_fixed, spark.table("default.orders").schema)
df_orders_fixed.writeTo("default.orders").overwritePartitions()

print("✅ Overwrote the 2025-06-15 partition (order 1002 is gone, 2025-06-16 untouched)")
spark.table("default.orders").orderBy("order_id").show(1000, False)

## Orders table maintenance

Iceberg procedures, all called as `CALL spark_catalog.system.<procedure>(...)`: [docs](https://iceberg.apache.org/docs/latest/spark-procedures/)

### Fragment the table

Six single-row commits, so the compaction procedures have something to do.

In [ ]:
orders_schema = spark.table("default.orders").schema

for i in range(6):
    row = [(
        1100 + i,
        f"Small Writer {i}",
        Decimal("15.00") + i,
        datetime(2025, 6, 17, 8 + i, 30),
        ("Tel Aviv", "IL"),
        "new",
    )]
    spark.createDataFrame(row, orders_schema).writeTo("default.orders").append()

### `create_changelog_view`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#create_changelog_view)

In [ ]:
snapshot_ids = [r[0] for r in spark.sql(
    "SELECT snapshot_id FROM default.orders.snapshots ORDER BY committed_at"
).collect()]

spark.sql(f"""
    CALL spark_catalog.system.create_changelog_view(
        table          => 'default.orders',
        options        => map('start-snapshot-id', '{snapshot_ids[0]}', 'end-snapshot-id', '{snapshot_ids[-1]}'),
        changelog_view => 'orders_changes'
    )
""").show(truncate=False)

spark.sql("""
    SELECT order_id, customer_name, amount, status, _change_type, _change_ordinal
    FROM orders_changes ORDER BY _change_ordinal, order_id
""").show(1000, False)

### Row-level DML

Merge-on-read, so these write the position delete files that the next procedure compacts.

In [ ]:
spark.sql("UPDATE default.orders SET status = 'delivered' WHERE order_id = 1004")

spark.sql("DELETE FROM default.orders WHERE order_id = 1101")

### `rewrite_position_delete_files`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#rewrite_position_delete_files)

In [ ]:
spark.sql("""
    CALL spark_catalog.system.rewrite_position_delete_files(
        table   => 'default.orders',
        options => map('rewrite-all', 'true')
    )
""").show(truncate=False)

### `rewrite_data_files` (binpack, zorder)

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#rewrite_data_files)

In [ ]:
spark.sql("""
    CALL spark_catalog.system.rewrite_data_files(
        table    => 'default.orders',
        strategy => 'binpack',
        options  => map('min-input-files', '2', 'rewrite-all', 'true')
    )
""").show(truncate=False)

spark.sql("""
    CALL spark_catalog.system.rewrite_data_files(
        table      => 'default.orders',
        strategy   => 'sort',
        sort_order => 'zorder(order_id, status)',
        options    => map('rewrite-all', 'true')
    )
""").show(truncate=False)

### `rewrite_manifests`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#rewrite_manifests)

In [ ]:
spark.sql("""
    CALL spark_catalog.system.rewrite_manifests(
        table       => 'default.orders',
        use_caching => false
    )
""").show(truncate=False)

### `compute_table_stats`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#compute_table_stats)

In [ ]:
spark.sql("""
    CALL spark_catalog.system.compute_table_stats(
        table   => 'default.orders',
        columns => array('order_id', 'customer_name', 'status')
    )
""").show(truncate=False)

### `compute_partition_stats`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#compute_partition_stats)

In [ ]:
spark.sql("CALL spark_catalog.system.compute_partition_stats(table => 'default.orders')").show(truncate=False)

### `expire_snapshots`

[docs](https://iceberg.apache.org/docs/latest/spark-procedures/#expire_snapshots)

In [ ]:
# Nessie creates tables with gc.enabled = false, which blocks any procedure that deletes files
spark.sql("ALTER TABLE default.orders SET TBLPROPERTIES ('gc.enabled' = 'true')")

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

spark.sql(f"""
    CALL spark_catalog.system.expire_snapshots(
        table                  => 'default.orders',
        older_than             => TIMESTAMP '{now}',
        retain_last            => 7,
        max_concurrent_deletes => 4,
        clean_expired_metadata => true
    )
""").show(truncate=False)

## Payments table

### 1. Create table, partitioned by `days(payment_ts)`

Another plain table: flat columns plus a small struct for the card used.

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.payments")
spark.sql("""
    CREATE TABLE default.payments (
        payment_id  BIGINT,
        payer       STRING,
        amount      DECIMAL(10,2),
        payment_ts  TIMESTAMP,
        card        STRUCT<brand: STRING, last4: STRING>
    )
    USING iceberg
    PARTITIONED BY (days(payment_ts))
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    """)
print("✅ Created table `default.payments`, partitioned by days(payment_ts)")

spark.table("default.payments").printSchema()

### 2. First write: append 3 payments

In [ ]:
from datetime import datetime
from decimal import Decimal

payments_day1 = [
    (5001, "Ada Lovelace", Decimal("120.50"), datetime(2025, 6, 15, 9, 35), ("visa", "4242")),
    (5002, "Grace Hopper", Decimal("89.99"), datetime(2025, 6, 15, 14, 10), ("mastercard", "8210")),
    (5003, "Alan Turing", Decimal("249.00"), datetime(2025, 6, 15, 18, 50), ("visa", "1717")),
]

df_payments = spark.createDataFrame(payments_day1, spark.table("default.payments").schema)
df_payments.writeTo("default.payments").append()

print("✅ Appended 3 payments for 2025-06-15")
spark.table("default.payments").show(1000, False)

### 3. Schema change 1: add a `currency` column

In [ ]:
spark.sql("ALTER TABLE default.payments ADD COLUMNS (currency STRING)")
print("✅ Added column: currency")

spark.table("default.payments").printSchema()
spark.table("default.payments").show(1000, False)

### 4. Second write: append 2 more payments, now with `currency`

The rows written before the change keep a NULL `currency`, no data is rewritten.

In [ ]:
payments_day2 = [
    (5004, "Katherine Johnson", Decimal("35.00"), datetime(2025, 6, 16, 10, 20), ("amex", "3005"), "USD"),
    (5005, "Ada Lovelace", Decimal("410.75"), datetime(2025, 6, 16, 16, 45), ("visa", "4242"), "GBP"),
]

df_payments_day2 = spark.createDataFrame(payments_day2, spark.table("default.payments").schema)
df_payments_day2.writeTo("default.payments").append()

print("✅ Appended 2 payments for 2025-06-16")
spark.table("default.payments").show(1000, False)

### 5. Schema change 2: rename `payer` to `payer_name`

In [ ]:
spark.sql("ALTER TABLE default.payments RENAME COLUMN payer TO payer_name")
print("✅ Renamed column: payer → payer_name")

spark.table("default.payments").printSchema()

### 6. Third write: overwrite only the 2025-06-15 partition

In [ ]:
payments_day1_fixed = [
    (5001, "Ada Lovelace", Decimal("130.50"), datetime(2025, 6, 15, 9, 35), ("visa", "4242"), "GBP"),
    (5003, "Alan Turing", Decimal("249.00"), datetime(2025, 6, 15, 18, 50), ("visa", "1717"), "GBP"),
]

df_payments_fixed = spark.createDataFrame(payments_day1_fixed, spark.table("default.payments").schema)
df_payments_fixed.writeTo("default.payments").overwritePartitions()

print("✅ Overwrote the 2025-06-15 partition (payment 5002 is gone, 2025-06-16 untouched)")
spark.table("default.payments").orderBy("payment_id").show(1000, False)

## Users table

### Create an empty table

Nothing is written to this one, it is created and left empty (no rows, no snapshots).

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.users")
spark.sql("""
    CREATE TABLE default.users (
        user_id     INT,
        user_name   STRING,
        email       STRING,
        created_at  TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (days(created_at))
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    """)
print("✅ Created empty table `default.users`, partitioned by days(created_at)")

spark.table("default.users").printSchema()
print(f"Row count: {spark.table('default.users').count()}")